# 0. Data Processing

Notebook rewrite of `code/0_data_processing`. It preserves the original preprocessing functions while grouping them into a clearer OpenAlex -> pickle workflow.

In [ ]:

from collections import defaultdict
from pathlib import Path
import ast, csv, gzip, json, pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
OPENALEX_ROOT = DATA_ROOT / "Openalex_2025"
SNAPSHOT_ROOT = OPENALEX_ROOT / "snapshot"
CSV_ROOT = OPENALEX_ROOT / "csv-files"
PICKLE_ROOT = OPENALEX_ROOT / "pickles"
SCIMAGO_ROOT = DATA_ROOT / "scimagojr"
WOS_ROOT = DATA_ROOT / "wos_core_collection"
for folder in [CSV_ROOT, PICKLE_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

def read_big_csv(file_path, **kwargs):
    chunksize = kwargs.pop("chunksize", 1_000_000)
    return pd.concat(pd.read_csv(file_path, chunksize=chunksize, **kwargs), ignore_index=True)

def dump_pickle(obj, name):
    with open(PICKLE_ROOT / name, "wb") as fh:
        pickle.dump(obj, fh, pickle.HIGHEST_PROTOCOL)

def load_pickle(name):
    with open(PICKLE_ROOT / name, "rb") as fh:
        return pickle.load(fh)

def openalex_id(entity):
    raw = (entity or {}).get("id") if isinstance(entity, dict) else entity
    return str(raw).rsplit("/", 1)[-1] if raw else None

def rows_to_csv(rows, out_path, columns):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(out_path, "wt", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def iter_jsonl_gz(entity_name, files_per_entity=None):
    files = sorted((SNAPSHOT_ROOT / "data" / entity_name).glob("**/*.gz"))
    if files_per_entity is not None:
        files = files[:files_per_entity]
    for file in files:
        with gzip.open(file, "rt", encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    yield json.loads(line)


## Flatten OpenAlex Snapshot

Condensed rewrite of `flatten-openalex-jsonl.py`, keeping the downstream CSV tables used by the project.

In [ ]:

def flatten_sources(files_per_entity=None):
    rows, ids = [], []
    for s in iter_jsonl_gz("sources", files_per_entity):
        sid = openalex_id(s)
        rows.append({"id": sid, "display_name": s.get("display_name"), "type": s.get("type"), "issn_l": s.get("issn_l"), "issn": repr(s.get("issn")), "works_count": s.get("works_count"), "cited_by_count": s.get("cited_by_count")})
        ids.append({"source_id": sid, "issn": repr(s.get("issn"))})
    rows_to_csv(rows, CSV_ROOT / "sources.csv.gz", ["id", "display_name", "type", "issn_l", "issn", "works_count", "cited_by_count"])
    rows_to_csv(ids, CSV_ROOT / "sources_ids.csv.gz", ["source_id", "issn"])

def flatten_institutions(files_per_entity=None):
    rows, geo_rows = [], []
    for inst in iter_jsonl_gz("institutions", files_per_entity):
        iid, geo = openalex_id(inst), inst.get("geo") or {}
        rows.append({"id": iid, "display_name": inst.get("display_name"), "country_code": inst.get("country_code"), "type": inst.get("type")})
        geo_rows.append({"institution_id": iid, "country_code": geo.get("country_code") or inst.get("country_code")})
    rows_to_csv(rows, CSV_ROOT / "institutions.csv.gz", ["id", "display_name", "country_code", "type"])
    rows_to_csv(geo_rows, CSV_ROOT / "institutions_geo.csv.gz", ["institution_id", "country_code"])

def flatten_topics(files_per_entity=None):
    rows = []
    for t in iter_jsonl_gz("topics", files_per_entity):
        rows.append({"id": openalex_id(t), "display_name": t.get("display_name"), "subfield_id": openalex_id(t.get("subfield")), "subfield_display_name": (t.get("subfield") or {}).get("display_name"), "field_id": openalex_id(t.get("field")), "field_display_name": (t.get("field") or {}).get("display_name"), "domain_id": openalex_id(t.get("domain")), "domain_display_name": (t.get("domain") or {}).get("display_name")})
    rows_to_csv(rows, CSV_ROOT / "topics.csv.gz", ["id", "display_name", "subfield_id", "subfield_display_name", "field_id", "field_display_name", "domain_id", "domain_display_name"])

def flatten_works(files_per_entity=None):
    works, authorships, refs, topics, locations, primary_locations, oa_rows = [], [], [], [], [], [], []
    for w in iter_jsonl_gz("works", files_per_entity):
        wid = openalex_id(w)
        works.append({"id": wid, "doi": w.get("doi"), "title": w.get("title"), "publication_year": w.get("publication_year"), "type": w.get("type")})
        oa_rows.append({"work_id": wid, "is_oa": (w.get("open_access") or {}).get("is_oa")})
        primary = w.get("primary_location") or {}
        primary_locations.append({"work_id": wid, "source_id": openalex_id((primary.get("source") or {})), "landing_page_url": primary.get("landing_page_url")})
        for loc in w.get("locations") or []:
            locations.append({"work_id": wid, "source_id": openalex_id((loc.get("source") or {})), "is_oa": loc.get("is_oa")})
        for a in w.get("authorships") or []:
            author, insts = a.get("author") or {}, a.get("institutions") or [None]
            for inst in insts:
                authorships.append({"work_id": wid, "author_id": openalex_id(author), "author_position": a.get("author_position"), "is_corresponding": a.get("is_corresponding"), "institution_id": openalex_id(inst) if inst else None})
        refs.extend({"work_id": wid, "referenced_work_id": openalex_id(r)} for r in (w.get("referenced_works") or []))
        topics.extend({"work_id": wid, "topic_id": openalex_id(t), "score": t.get("score")} for t in (w.get("topics") or []))
    rows_to_csv(works, CSV_ROOT / "works.csv.gz", ["id", "doi", "title", "publication_year", "type"])
    rows_to_csv(authorships, CSV_ROOT / "works_authorships.csv.gz", ["work_id", "author_id", "author_position", "is_corresponding", "institution_id"])
    rows_to_csv(refs, CSV_ROOT / "works_referenced_works.csv.gz", ["work_id", "referenced_work_id"])
    rows_to_csv(topics, CSV_ROOT / "works_topics.csv.gz", ["work_id", "topic_id", "score"])
    rows_to_csv(locations, CSV_ROOT / "works_locations.csv.gz", ["work_id", "source_id", "is_oa"])
    rows_to_csv(primary_locations, CSV_ROOT / "works_primary_locations.csv.gz", ["work_id", "source_id", "landing_page_url"])
    rows_to_csv(oa_rows, CSV_ROOT / "works_open_access.csv.gz", ["work_id", "is_oa"])

def flatten_openalex_snapshot(files_per_entity=None):
    flatten_sources(files_per_entity)
    flatten_institutions(files_per_entity)
    flatten_topics(files_per_entity)
    flatten_works(files_per_entity)

# flatten_openalex_snapshot(files_per_entity=1)
# flatten_openalex_snapshot()


## Build Analysis Pickles

Functional rewrite of `PY1` through `PY4` and the WoS ISSN script.

In [ ]:

def build_citation_matrix():
    edge_df = read_big_csv(CSV_ROOT / "works_referenced_works.csv.gz", compression="gzip", usecols=["work_id", "referenced_work_id"], dtype={"work_id": "string", "referenced_work_id": "string"}).dropna()
    citing_raw, cited_raw = edge_df["work_id"].tolist(), edge_df["referenced_work_id"].tolist()
    works = read_big_csv(CSV_ROOT / "works.csv.gz", compression="gzip", usecols=["id"], dtype={"id": "string"})
    paper_id = {pid: i for i, pid in enumerate(sorted(set(citing_raw) | set(cited_raw) | set(works["id"].dropna())))}
    dump_pickle(paper_id, "Paper_newID.pickle")
    citing = np.fromiter((paper_id[x] for x in citing_raw), dtype=np.int64)
    cited = np.fromiter((paper_id[x] for x in cited_raw), dtype=np.int64)
    order = np.lexsort((cited, citing))
    citing, cited = citing[order], cited[order]
    keep = np.ones(len(citing), dtype=bool)
    keep[1:] = (citing[1:] != citing[:-1]) | (cited[1:] != cited[:-1])
    matrix = csr_matrix((np.ones(keep.sum(), dtype=np.int8), (citing[keep], cited[keep])), shape=(len(paper_id), len(paper_id)))
    dump_pickle(matrix, "citation_matrix_csr.pickle")
    dump_pickle(matrix.indptr, "citation_matrix_csr_indptr.pickle")
    dump_pickle(matrix.indices, "citation_matrix_csr_indices.pickle")
    return matrix

def derive_paper_properties():
    paper_id = load_pickle("Paper_newID.pickle")
    paper_df = pd.DataFrame({"work_id": list(paper_id.keys()), "work_id_new": list(paper_id.values())})
    works = read_big_csv(CSV_ROOT / "works.csv.gz", compression="gzip", usecols=["id", "publication_year", "type"], dtype={"id": "string", "type": "string"}).rename(columns={"id": "work_id"})
    works_new = paper_df.merge(works, on="work_id", how="inner")
    dump_pickle(works_new.dropna(subset=["work_id_new", "publication_year"]).drop_duplicates(["work_id_new", "publication_year"]).set_index("work_id_new")["publication_year"].to_dict(), "Paper_year.pickle")
    auth = read_big_csv(CSV_ROOT / "works_authorships.csv.gz", compression="gzip", usecols=["work_id", "author_id", "author_position", "is_corresponding", "institution_id"], dtype={"work_id": "string", "author_id": "string", "author_position": "string", "institution_id": "string"})
    auth = paper_df.merge(auth, on="work_id", how="inner").drop(columns=["work_id"])
    inst = read_big_csv(CSV_ROOT / "institutions.csv.gz", compression="gzip", usecols=["id", "country_code"], dtype={"id": "string", "country_code": "string"}).rename(columns={"id": "institution_id"})
    geo = read_big_csv(CSV_ROOT / "institutions_geo.csv.gz", compression="gzip", usecols=["institution_id", "country_code"], dtype={"institution_id": "string", "country_code": "string"})
    auth_country = auth.merge(pd.concat([inst, geo]).dropna().drop_duplicates("institution_id"), on="institution_id", how="inner")
    dump_pickle(auth_country.groupby("work_id_new")["country_code"].apply(list).to_dict(), "Paper_country_all_aff.pickle")
    first = auth_country[auth_country.author_position == "first"].drop_duplicates(["work_id_new", "author_position"]).dropna(subset=["country_code"])
    dump_pickle(first.set_index("work_id_new")["country_code"].to_dict(), "Paper_country_1st_aff.pickle")
    corr = auth_country[auth_country.is_corresponding == True].drop_duplicates(["work_id_new", "author_id"]).drop_duplicates(["work_id_new", "is_corresponding"], keep="last").dropna(subset=["country_code"])
    dump_pickle(corr.set_index("work_id_new")["country_code"].to_dict(), "Paper_country_corr_aff.pickle")
    works_topics = read_big_csv(CSV_ROOT / "works_topics.csv.gz", compression="gzip", usecols=["work_id", "topic_id", "score"], dtype={"work_id": "string", "topic_id": "string"})
    works_topics = paper_df.merge(works_topics, on="work_id", how="inner").drop(columns=["work_id"])
    for tag in ["domain", "field", "subfield"]:
        col = f"{tag}_display_name"
        topics = read_big_csv(CSV_ROOT / "topics.csv.gz", compression="gzip", usecols=["id", col], dtype={"id": "string", col: "string"}).rename(columns={"id": "topic_id"})
        info = works_topics.merge(topics, on="topic_id", how="inner").dropna(subset=["work_id_new", col, "score"])
        weights = info.groupby(["work_id_new", col])["score"].sum().div(info.groupby("work_id_new")["score"].sum(), level="work_id_new").reset_index(name="weight").sort_values(["work_id_new", "weight"], ascending=[True, False])
        dump_pickle(weights.groupby("work_id_new").apply(lambda g: {tag: g[col].tolist(), "weight": g["weight"].tolist()}).to_dict(), f"Paper_{tag}_weight_list.pickle")
        dump_pickle(weights.drop_duplicates("work_id_new").set_index("work_id_new")[col].to_dict(), f"Paper_{tag}_1st.pickle")
    oa = read_big_csv(CSV_ROOT / "works_open_access.csv.gz", compression="gzip", usecols=["work_id", "is_oa"], dtype={"work_id": "string"}).dropna()
    dump_pickle(paper_df.merge(oa, on="work_id", how="inner").drop_duplicates(["work_id_new", "is_oa"]).set_index("work_id_new")["is_oa"].to_dict(), "Paper_OA_tag.pickle")

def parse_list_cell(x):
    return x if pd.isna(x) else (ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x)

def build_paper_issn_list():
    sources = read_big_csv(CSV_ROOT / "sources_ids.csv.gz", compression="gzip", usecols=["source_id", "issn"], dtype={"source_id": "string", "issn": "string"}).dropna().drop_duplicates(["source_id", "issn"])
    sources["issn_list"] = sources["issn"].apply(parse_list_cell)
    works_sources = read_big_csv(CSV_ROOT / "works_locations.csv.gz", compression="gzip", usecols=["work_id", "source_id"], dtype={"work_id": "string", "source_id": "string"}).dropna().drop_duplicates(["work_id", "source_id"])
    paper_id = load_pickle("Paper_newID.pickle")
    paper_df = pd.DataFrame({"work_id": list(paper_id.keys()), "work_id_new": list(paper_id.values())})
    merged = paper_df.merge(works_sources.merge(sources.drop(columns=["issn"]), on="source_id", how="inner"), on="work_id", how="inner")
    dump_pickle(merged.drop(columns=["work_id"]).set_index("work_id_new")["issn_list"].to_dict(), "Paper_ISSN_list.pickle")

def build_wos_issn_table():
    files = [("Science Citation Index Expanded (SCIE).csv", "SCIE"), ("Social Sciences Citation Index (SSCI).csv", "SSCI"), ("Arts & Humanities Citation Index (AHCI).csv", "AHCI"), ("Emerging Sources Citation Index (ESCI).csv", "ESCI")]
    frames = []
    for name, index_type in files:
        df = pd.read_csv(WOS_ROOT / name)
        df["Index Type"] = index_type
        frames.append(df)
    info = pd.concat(frames, ignore_index=True)
    issn = info.drop(columns=["eISSN"], errors="ignore").dropna(subset=["ISSN"]).drop_duplicates("ISSN")
    eissn = info.drop(columns=["ISSN"], errors="ignore").dropna(subset=["eISSN"]).drop_duplicates("eISSN").rename(columns={"eISSN": "ISSN"})
    out = pd.concat([issn, eissn], ignore_index=True).drop_duplicates("ISSN")
    out.to_csv(PICKLE_ROOT / "wos2025_Journal_ISSN_all.csv", index=False)
    return out

def build_paper_sjr_quartile():
    issn_rank = defaultdict(dict)
    for year in range(1999, 2026):
        df = pd.read_csv(SCIMAGO_ROOT / f"scimagojr_{year}.csv", sep=";")
        df = df[df["SJR Best Quartile"] != "-"]
        for issns, rank in zip(df["Issn"], df["SJR Best Quartile"]):
            for raw in str(issns).split(", "):
                issn_rank[raw[:4] + "-" + raw[4:]][year] = rank
    rank_df = pd.DataFrame([{"ISSN": i, "year": y, "rank": r} for i, yr in issn_rank.items() for y, r in yr.items()])
    paper_year = load_pickle("Paper_year.pickle")
    paper_year_df = pd.DataFrame({"PaperID": list(paper_year.keys()), "year": list(paper_year.values())}).query("1900 <= year <= 2025")
    paper_issn = load_pickle("Paper_ISSN_list.pickle")
    paper_issn_df = pd.DataFrame({"PaperID": list(paper_issn.keys()), "ISSN": list(paper_issn.values())}).explode("ISSN").reset_index(drop=True)
    merged = paper_issn_df.merge(paper_year_df, on="PaperID", how="inner").merge(rank_df, on=["ISSN", "year"], how="inner")
    dump_pickle(merged[["PaperID", "rank"]].set_index("PaperID").to_dict()["rank"], "Paper_ISSN_SJR.pickle")

# Recommended full order:
# flatten_openalex_snapshot()
# build_citation_matrix()
# derive_paper_properties()
# build_paper_issn_list()
# build_wos_issn_table()
# build_paper_sjr_quartile()
